# Team Toffoli

Our designs, tests and results for the DQC Toffoli challenge. The rules and background are in `Toffoli_Hardware_Hackathon_Challenge.ipynb` in this folder. Run this notebook with the "Python (toffoli)" kernel.

## 0. Setup

Team values live in `.env` at the repository root, which git ignores. The first cell copies them into the environment and prints only whether each one is set. It runs before the challenge package is imported, because the mystery-backend client reads its URL when it is imported.

Every hardware submission is logged to `submissions/queue_log.json` (also ignored by git), so each number we present traces back to a queue ID.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def load_env(path):
    """Copy KEY=VALUE lines from a .env file into os.environ (values already set win)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, sep, value = line.partition("=")
        if sep and value.strip():
            os.environ.setdefault(key.strip(), value.strip())


load_env(ROOT / ".env")
for key in ("TEAM_ID", "TEAM_SECRET", "MYSTERY_BACKEND_URL", "MYSTERY_BACKEND_TOKEN"):
    print(f"{key:22s} set: {bool(os.environ.get(key))}")

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, qasm2
from qiskit.circuit.library import SXGate
from qiskit.quantum_info import Operator, Statevector
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGatesDecomposition

from toffoli_challenge import (
    TOFFOLI_INPUTS,
    baseline_native_toffoli,
    compare_to_toffoli,
    validate_submission_circuit,
    with_input_preparation,
)

RESULTS_DIR = ROOT / "returned_results"
QUEUE_LOG = ROOT / "submissions" / "queue_log.json"


def log_submission(receipt, **details):
    """Append one hardware submission (its queue ID and what it tested) to the local log."""
    QUEUE_LOG.parent.mkdir(exist_ok=True)
    log = json.loads(QUEUE_LOG.read_text()) if QUEUE_LOG.exists() else []
    entry = {"queue_id": receipt["submission"]["queue_id"],
             "logged_at": datetime.now().isoformat(timespec="seconds"), **details}
    log.append(entry)
    QUEUE_LOG.write_text(json.dumps(log, indent=2))
    return entry


print("repository root:", ROOT.name)

## 1. Designs

A Toffoli needs at least five two-qubit gates (Yu, Duan and Ying, *Phys. Rev. A* 88, 010304, 2013), so the baseline's five `rxx` gates cannot be cut. Two things can still change: the size of the `rxx` angles, and the number of noisy one-qubit gates around them. `rz` is done in software and is free, so only `rx`, `ry` and `rxx` cost anything.

D0 is the starter kit's baseline. It is Qiskit's compilation, with five `rxx` and a total angle of 2.25π.

D1 uses the construction from Barenco et al., *Phys. Rev. A* 52, 3457 (1995): a controlled-√X from c2 to t, a CNOT from c1 to c2, a controlled-√X† from c2 to t, the same CNOT again, and a controlled-√X from c1 to t. When both controls are 1, the target gets √X twice, which is X. When exactly one control is 1, the target gets one √X and one √X†, which cancel. A controlled-√X needs only `rxx(π/4)`, half the angle of a CNOT's `rxx(π/2)`, so the total angle is 3 × π/4 + 2 × π/2 = 1.75π.

D2 is not a new circuit. It is D0 or D1 with each `rxx` angle adjusted to cancel an over-rotation that we measure on the hardware (Section 4).

### Building each block from one `rxx`

A controlled-√X is exp(iπ/8 (I − Z)⊗(I − X)). When the control is 0 the exponent is 0. When it is 1 the target gets exp(iπ/4 (I − X)), which is √X.

Multiplying out the exponent leaves one two-qubit term, exp(iπ/8 Z⊗X), plus one-qubit rotations: `rz(π/4)` on the control and `rx(π/4)` on the target. An `ry(π/2)` on the control turns its Z into X, so the two-qubit term becomes exp(iπ/8 X⊗X), which is `rxx(-π/4)`.

A CNOT is exp(iπ/4 (I − Z)⊗(I − X)), so the same steps give `rxx(-π/2)`. The next cell checks each block against Qiskit's own gate.

In [ ]:
C1, C2, T = 2, 1, 0  # challenge wires: q[2] = c1, q[1] = c2, q[0] = t


def controlled_sqrt_x(qc, control, target, inverse=False):
    """Controlled-sqrt(X), exact up to global phase, built from one rxx(-pi/4)."""
    s = -1 if inverse else 1
    qc.ry(np.pi / 2, control)
    qc.rxx(-s * np.pi / 4, control, target)
    qc.ry(-np.pi / 2, control)
    qc.rz(s * np.pi / 4, control)
    qc.rx(s * np.pi / 4, target)


def cnot_native(qc, control, target):
    """CNOT, exact up to global phase, built from one rxx(-pi/2)."""
    qc.ry(np.pi / 2, control)
    qc.rxx(-np.pi / 2, control, target)
    qc.ry(-np.pi / 2, control)
    qc.rz(np.pi / 2, control)
    qc.rx(np.pi / 2, target)


checks = [
    ("controlled-sqrt(X)", lambda qc: controlled_sqrt_x(qc, 1, 0), lambda qc: qc.append(SXGate().control(1), [1, 0])),
    ("controlled-sqrt(X)^dagger", lambda qc: controlled_sqrt_x(qc, 1, 0, inverse=True),
     lambda qc: qc.append(SXGate().inverse().control(1), [1, 0])),
    ("CNOT", lambda qc: cnot_native(qc, 1, 0), lambda qc: qc.cx(1, 0)),
]
for name, build, build_reference in checks:
    block, reference = QuantumCircuit(2), QuantumCircuit(2)
    build(block)
    build_reference(reference)
    print(f"{name:26s} matches Qiskit's gate: {Operator(block).equiv(Operator(reference))}")

### The designs and their costs

Between `rxx` gates, the one-qubit gates on each wire can be merged into one rotation, written rz · ry · rz. Since `rz` is free, each merged run costs at most one `ry`. `compress_1q` does this with Qiskit's `Optimize1qGatesDecomposition` pass and leaves every `rxx` as it is.

In [ ]:
def small_angle_toffoli():
    """D1: the Barenco et al. construction with controls c1 = q[2], c2 = q[1] and target t = q[0]."""
    qc = QuantumCircuit(3)
    controlled_sqrt_x(qc, C2, T)
    cnot_native(qc, C1, C2)
    controlled_sqrt_x(qc, C2, T, inverse=True)
    cnot_native(qc, C1, C2)
    controlled_sqrt_x(qc, C1, T)
    return qc


def compress_1q(circuit):
    """Merge each run of one-qubit gates into rz-ry-rz, so each run costs at most one ry."""
    return PassManager([Optimize1qGatesDecomposition(basis=["rz", "ry"])]).run(circuit)


def costs(circuit):
    """Return (number of rxx, total |rxx angle| / pi, number of rx and ry)."""
    angles = [float(i.operation.params[0]) for i in circuit.data if i.operation.name == "rxx"]
    costly_1q = sum(1 for i in circuit.data if i.operation.name in ("rx", "ry"))
    return len(angles), sum(map(abs, angles)) / np.pi, costly_1q


designs = {
    "D0": baseline_native_toffoli(),
    "D0c": compress_1q(baseline_native_toffoli()),
    "D1": small_angle_toffoli(),
    "D1c": compress_1q(small_angle_toffoli()),
}

print(f"{'design':7s} {'rxx':>4s} {'total angle':>12s} {'rx + ry':>8s} {'ideal process fidelity':>23s}")
for name, circuit in designs.items():
    validate_submission_circuit(circuit)
    n_rxx, total_angle, costly_1q = costs(circuit)
    _, process_fidelity, _ = compare_to_toffoli(circuit)
    print(f"{name:7s} {n_rxx:4d} {total_angle:10.3f}pi {costly_1q:8d} {process_fidelity:23.6f}")

In [ ]:
def compensate(circuit, offset=0.0, scale=1.0):
    """Copy a circuit with each rxx angle theta replaced by scale * theta - offset.

    A fixed over-rotation (theta -> theta + offset) is cancelled by offset.
    An over-rotation that grows with the angle (theta -> theta * (1 + e)) is cancelled by scale = 1 / (1 + e).
    """
    out = circuit.copy_empty_like()
    for instruction in circuit.data:
        if instruction.operation.name == "rxx":
            out.rxx(scale * float(instruction.operation.params[0]) - offset, *instruction.qubits)
        else:
            out.append(instruction)
    return out


display(designs["D1c"].draw("mpl", fold=-1))